In [17]:
using System;
using System.Collections.Generic;

public abstract class Component
{
    public string Model {get; set;}
    public decimal Price { get; set; }
    public bool IsInstalled { get; protected set; }

    public Component(string model, decimal price)
    {
        Model = model;
        Price = price;
        IsInstalled = false;
    }

    public virtual string GetInfo()
    {
        return $"{Model} - ${Price}";
    }

    public override string ToString()
    {
        return GetInfo();
    }

    public static bool operator ==(Component comp1, Component comp2)
    {
        if (ReferenceEquals(comp1, comp2)) return true;
        if (comp1 is null || comp2 is null) return false;
        
        return comp1.Model == comp2.Model && 
               comp1.Price == comp2.Price;
    }

    public override bool Equals(object obj)
    {
        return this == (obj as Component);
    }

    public static bool operator !=(Component comp1, Component comp2)
    {
        return !(comp1 == comp2);
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Model, Price);
    }


}

public interface IInstallable
{
    void Install();
    bool IsInstalled { get; }
}

public class CPU : Component, IInstallable
{
    public int Cores { get; set; }
    public double Frequency { get; set; }

    public CPU(string model, decimal price, int cores, double frequency) 
        : base(model, price)
    {
        Cores = cores;
        Frequency = frequency;
    }

    public CPU() : this("Basic CPU", 99.99m, 2, 2.0) {}

    public override string GetInfo()
    {
        return $"{base.GetInfo()} | {Cores} cores, {Frequency}GHz";
    }

     public void Install()
    {
        if (!IsInstalled)
        {
            IsInstalled = true;
            Console.WriteLine($"CPU {Model} установлен в сокет");
        }
        else
        {
            Console.WriteLine($"CPU {Model} уже установлен");
        }
    }
}

public class RAM : Component, IInstallable
{
    public int Capacity { get; set; }
    public string Type { get; set; }

    public RAM(string model, decimal price, int capacity, string type) 
        : base(model, price)
    {
        Capacity = capacity;
        Type = type;
    }

    public RAM() : this("Basic RAM", 39.99m, 4, "DDR4") {}

    public override string GetInfo()
    {
        return $"{base.GetInfo()} | {Capacity}GB {Type}";
    }

    public void Install()
    {
        if (!IsInstalled)
        {
            IsInstalled = true;
            Console.WriteLine($"RAM {Model} установлена в DIMM слот");
        }
        else
        {
            Console.WriteLine($"RAM {Model} уже установлена");
        }
    }
} 

public class GPU : Component, IInstallable
{
    public int VRAM { get; set; }
    public string Interface { get; set; }

    public GPU(string model, decimal price, int vram, string interfaceType) 
        : base(model, price)
    {
        VRAM = vram;
        Interface = interfaceType;
    }

    public GPU() : this("Basic GPU", 149.99m, 2, "PCIe") {}

    public override string GetInfo()
    {
        return $"{base.GetInfo()} | {VRAM}GB VRAM, {Interface}";
    }

    public void Install()
    {
        if (!IsInstalled)
        {
            IsInstalled = true;
            Console.WriteLine($"GPU {Model} установлена в  {Interface} слот");
        }
        else
        {
            Console.WriteLine($"GPU {Model} is already installed");
        }
    }
}

public class HDD : Component, IInstallable
{
    public int Capacity { get; set; }
    public string FormFactor { get; set; }

    public HDD(string model, decimal price, int capacity, string formFactor) 
        : base(model, price)
    {
        Capacity = capacity;
        FormFactor = formFactor;
    }

    public HDD() : this("Basic HDD", 49.99m, 500, "3.5\"") {}

    public override string GetInfo()
    {
        return $"{base.GetInfo()} | {Capacity}GB, {FormFactor}";
    }

    public void Install()
    {
        if (!IsInstalled)
        {
            IsInstalled = true;
            Console.WriteLine($"HDD {Model} установлен в {FormFactor} порт");
        }
        else
        {
            Console.WriteLine($"HDD {Model} уже установлен");
        }
    }
}

public class Computer 
{
    private List<Component> _components = new List<Component>();

    public string Name { get; set; }
    public int ComponentCount => _components.Count;

    public Computer(string name)
    {
        Name = name;
        _components = new List<Component>();
    }

     public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        if (component == null)
            throw new ArgumentNullException(nameof(component));

        if (!_components.Contains(component))
        {
            _components.Add(component);
            component.Install();
            Console.WriteLine($"{component.GetType().Name} '{component.Model}' добавлен в компьютер '{Name}'");
        }
        else
        {
            Console.WriteLine($"Компонент '{component.Model}' уже находится в компьютере");
        }
    }

    public bool RemoveComponent<T>(T component) where T : Component, IInstallable
    {
        if (component == null)
            return false;

        bool removed = _components.Remove(component);
        if (removed)
        {
            Console.WriteLine($"{component.GetType().Name} '{component.Model}' удален из компьютера '{Name}'");
        }
        return removed;
    }

    public decimal GetTotalPrice()
    {
        return _components.Sum(component => component.Price);
    }

    public void DisplayComponents()
    {
        Console.WriteLine($"\nКомпоненты в компьютере '{Name}':");
        if (_components.Count == 0)
        {
            Console.WriteLine("Компоненты не установлены");
            return;
        }

        foreach (var component in _components)
        {
            Console.WriteLine($"  - {component.GetInfo()}");
        }
        Console.WriteLine($"Общая стоимость: ${GetTotalPrice()}");
    }
}

public static class ComputerFactory
{
    // базавая
    public static Computer CreateBasicComputer<TCPU, TRAM, THDD>(string name) 
        where TCPU : CPU, new()
        where TRAM : RAM, new()
        where THDD : HDD, new()
    {
        var computer = new Computer(name);
        
        var cpu = new TCPU();
        var ram = new TRAM();
        var hdd = new THDD();

        computer.AddComponent(cpu);
        computer.AddComponent(ram);
        computer.AddComponent(hdd);

        Console.WriteLine($"Создана базовая конфигурация компьютера '{name}' с компонентами: {typeof(TCPU).Name}, {typeof(TRAM).Name}, {typeof(THDD).Name}");

        return computer;
    }

    // игровая
    public static Computer CreateGamingComputer<TCPU, TGPU, TRAM, THDD>(string name) 
        where TCPU : CPU, new()
        where TGPU : GPU, new()
        where TRAM : RAM, new()
        where THDD : HDD, new()
    {
        var computer = new Computer(name);
        
        var cpu = new TCPU();
        var gpu = new TGPU();
        var ram = new TRAM();
        var hdd = new THDD();

        computer.AddComponent(cpu);
        computer.AddComponent(gpu);
        computer.AddComponent(ram);
        computer.AddComponent(hdd);

        Console.WriteLine($"Создана игровая конфигурация компьютера '{name}' с компонентами: {typeof(TCPU).Name}, {typeof(TGPU).Name}, {typeof(TRAM).Name}, {typeof(THDD).Name}");

        return computer;
    }

    // кастомка
    public static Computer CreateCustomComputer(string name, params IInstallable[] components)
    {
        var computer = new Computer(name);
        
        foreach (var component in components)
        {
            switch (component)
            {
                case CPU cpu:
                    computer.AddComponent(cpu);
                    break;
                case RAM ram:
                    computer.AddComponent(ram);
                    break;
                case GPU gpu:
                    computer.AddComponent(gpu);
                    break;
                case HDD hdd:
                    computer.AddComponent(hdd);
                    break;
            }
        }

        Console.WriteLine($"Создана кастомная конфигурация '{name}' с {components.Length} компонентами");

        return computer;
    }
}

CPU cpu = new CPU("Intel i7", 299.99m, 8, 3.8);
RAM ram = new RAM("Corsair 16GB", 89.99m, 16, "DDR4");
GPU gpu = new GPU("NVIDIA RTX 4060", 399.99m, 8, "PCIe 4.0");
HDD hdd = new HDD("WD Blue 1TB", 49.99m, 1000, "3.5\"");

Computer myPC= new Computer("Мой ПК");
myPC.AddComponent(cpu);
myPC.AddComponent(gpu);
myPC.DisplayComponents();

Console.WriteLine(cpu.GetInfo());

CPU cpu1 = new CPU("Intel i7", 299.99m, 8, 3.8);
CPU cpu2 = new CPU("Intel i7", 299.99m, 8, 3.8);

Console.WriteLine(cpu1 == cpu2);
Console.WriteLine(cpu1.Equals(cpu2));

var basicPC = ComputerFactory.CreateBasicComputer<CPU, RAM, HDD>("Базовый ПК");
        basicPC.DisplayComponents();

var gamingPC = ComputerFactory.CreateGamingComputer<CPU, GPU, RAM, HDD>("Игровой ПК");
        gamingPC.DisplayComponents();

var customPC = ComputerFactory.CreateCustomComputer("Кастомный ПК", cpu, ram, gpu);
        customPC.DisplayComponents();


CPU Intel i7 установлен в сокет
CPU 'Intel i7' добавлен в компьютер 'Мой ПК'
GPU NVIDIA RTX 4060 установлена в  PCIe 4.0 слот
GPU 'NVIDIA RTX 4060' добавлен в компьютер 'Мой ПК'

Компоненты в компьютере 'Мой ПК':
  - Intel i7 - $299,99 | 8 cores, 3,8GHz
  - NVIDIA RTX 4060 - $399,99 | 8GB VRAM, PCIe 4.0
Общая стоимость: $699,98
Intel i7 - $299,99 | 8 cores, 3,8GHz
True
True
CPU Basic CPU установлен в сокет
CPU 'Basic CPU' добавлен в компьютер 'Базовый ПК'
RAM Basic RAM установлена в DIMM слот
RAM 'Basic RAM' добавлен в компьютер 'Базовый ПК'
HDD Basic HDD установлен в 3.5" порт
HDD 'Basic HDD' добавлен в компьютер 'Базовый ПК'
Создана базовая конфигурация компьютера 'Базовый ПК' с компонентами: CPU, RAM, HDD

Компоненты в компьютере 'Базовый ПК':
  - Basic CPU - $99,99 | 2 cores, 2GHz
  - Basic RAM - $39,99 | 4GB DDR4
  - Basic HDD - $49,99 | 500GB, 3.5"
Общая стоимость: $189,97
CPU Basic CPU установлен в сокет
CPU 'Basic CPU' добавлен в компьютер 'Игровой ПК'
GPU Basic GPU установлена